<div style='background:linear-gradient(135deg,#0a0a1a 0%,#1a1a2e 40%,#0f3460 100%);padding:28px;border-radius:10px'>
<h1 style='color:#fff;margin:0'>🛡️ FedShield — Federated Mule-Account Detection</h1>
<p style='color:#a8dadc;margin:6px 0 0'>Privacy-preserving <b>Federated Averaging (FedAvg)</b> across simulated banks · RBIH-NFPC Phase-2 · Kaggle free-tier (CPU, no GPU)</p>
</div>

**This is a corrected, reproducible rebuild.** It fixes the methodology of the earlier notebook:

| Earlier notebook | This notebook |
|---|---|
| "FedAvg" averaged the **probabilities** of 3 models re-trained from scratch each round with `random_state=rnd` — no state carried, so rounds were just random seeds (soft-voting ensemble, not federation). | **Real FedAvg**: one global parameter vector; each round every bank does local GD from the current global weights, and the **weight updates** are averaged and carried forward. Convergence is real. |
| Reported `iloc[-1]` (last round). Round 10's recall collapsed to 0.41, which *inflated* precision to 0.73 — the headline "0.38→0.73 precision" was a recall-collapse artifact. | Threshold-free headline metrics (**AUC, AUPRC**) plus **precision/F1 at a fixed-recall operating point** chosen on a validation split. Reported as **mean ± std over seeds**, not one lucky round. |
| Baselines (200 trees, early stopping) vs "federated" (100-tree 3-model ensemble) — unfair. | All comparisons use the **same model class**; a centralized model is shown as an explicit **upper bound**. |
| Differential privacy, evidence fusion, etc. claimed but **not implemented**. | DP implemented as **gradient-clipping + calibrated Gaussian noise**, with a measured privacy/utility curve. Everything not implemented is stated plainly in *Scope & honesty* below. |

<div style='background:#0d1117;border-left:4px solid #f0a500;padding:16px;border-radius:6px;margin-top:10px'>
<b style='color:#f0a500'>⚠️ Scope &amp; honesty.</b>
<span style='color:#ccc'>This notebook simulates a 3-bank federation by partitioning a <b>single</b> RBIH-NFPC dataset. It does <b>not</b> ingest live EFRM / AML / I4C / DPIP feeds — the 7-layer evidence-fusion architecture in the one-pager is design-stage, not run here. What <b>is</b> run end-to-end: federated feature standardisation (no raw rows shared), real FedAvg on a logistic model, differential-privacy noise, fair local/centralised baselines, and SHAP explainability.</span>
</div>

## How to run on Kaggle free tier
1. **Add data**: *Add Input* → search `rbih-nfpc-phase-2` (abhyudayrbih) → attach.
2. **Accelerator**: *None (CPU)* is enough. ~15–20 min cold (the transaction streaming dominates); modelling is seconds.
3. **Run All**. Heavy steps checkpoint to `/kaggle/working`, so re-runs skip straight to modelling.

In [5]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Setup, imports, robust dataset-path detection     ║
# ║  Run this every time the kernel restarts.                   ║
# ╚═══════════════════════════════════════════════════════════╝
import subprocess, sys, os, gc, glob, warnings, pickle
subprocess.run([sys.executable,'-m','pip','install','lightgbm','shap','--quiet'], capture_output=True)

import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, confusion_matrix,
                             precision_recall_curve)
warnings.filterwarnings('ignore')
plt.style.use('dark_background')
np.random.seed(42)

COLORS = {'red':'#e94560','blue':'#a8dadc','gold':'#f0a500','green':'#4caf50',
          'bankA':'#e94560','bankB':'#a8dadc','bankC':'#f0a500','fed':'#4caf50','cen':'#9b5de5'}
CKPT_DIR   = '/kaggle/working'
BATCH_SIZE = 10            # transaction files per streaming batch (conservative for free tier)
os.makedirs(CKPT_DIR, exist_ok=True)

# --- Auto-detect dataset root: find the folder that contains accounts.parquet ---
def find_base_path():
    hard = '/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2'
    if os.path.isfile(os.path.join(hard,'accounts.parquet')):
        return hard
    hits = glob.glob('/kaggle/input/**/accounts.parquet', recursive=True)
    if hits:
        return os.path.dirname(sorted(hits, key=len)[0])
    raise FileNotFoundError(
        'Could not find accounts.parquet under /kaggle/input. '
        'Attach the rbih-nfpc-phase-2 dataset via "Add Input".')

BASE_PATH = find_base_path()
print('✅ Setup complete')
print('📁 Dataset   :', BASE_PATH)
print('💾 Checkpoints:', CKPT_DIR, '\n')
for f in sorted(os.listdir(BASE_PATH)):
    fp = os.path.join(BASE_PATH, f)
    tag = f'{os.path.getsize(fp)/1e6:6.1f} MB' if os.path.isfile(fp) else '  <dir>'
    print(f'  {f:<40} {tag}')


✅ Setup complete
📁 Dataset   : /kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2
💾 Checkpoints: /kaggle/working 

  README.md                                   0.0 MB
  accounts-additional.parquet                 1.0 MB
  accounts.parquet                            7.1 MB
  branch.parquet                              0.3 MB
  customer_account_linkage.parquet            1.9 MB
  customers.parquet                           2.4 MB
  demographics.parquet                        4.8 MB
  product_details.parquet                     3.0 MB
  test_accounts.parquet                       0.4 MB
  train_labels.parquet                        0.7 MB
  transactions                               <dir>
  transactions_additional                    <dir>


In [6]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Load static files (checkpoint-aware)              ║
# ╚═══════════════════════════════════════════════════════════╝
FEAT_CKPT = os.path.join(CKPT_DIR, 'ckpt_features.parquet')

if os.path.exists(FEAT_CKPT):
    print('✅ SKIP — features checkpoint exists; go to Cell 5.')
else:
    print('📥 Loading static files...')
    accounts     = pd.read_parquet(f'{BASE_PATH}/accounts.parquet')
    customers    = pd.read_parquet(f'{BASE_PATH}/customers.parquet')
    demographics = pd.read_parquet(f'{BASE_PATH}/demographics.parquet')
    branch       = pd.read_parquet(f'{BASE_PATH}/branch.parquet')
    linkage      = pd.read_parquet(f'{BASE_PATH}/customer_account_linkage.parquet')
    products     = pd.read_parquet(f'{BASE_PATH}/product_details.parquet')
    acc_add      = pd.read_parquet(f'{BASE_PATH}/accounts-additional.parquet')
    labels       = pd.read_parquet(f'{BASE_PATH}/train_labels.parquet')
    for df_ in [accounts, customers, branch, linkage, products, acc_add]:
        for c in df_.select_dtypes('float64').columns: df_[c] = df_[c].astype('float32')
        for c in df_.select_dtypes('int64').columns:   df_[c] = pd.to_numeric(df_[c], downcast='integer')
    for nm, d in [('accounts',accounts),('customers',customers),('demographics',demographics),
                  ('branch',branch),('linkage',linkage),('product_details',products),
                  ('accounts-additional',acc_add),('train_labels',labels)]:
        print(f'  {nm:<20}: {d.shape}')
    print('✅ Static files loaded')


✅ SKIP — features checkpoint exists; go to Cell 5.


In [7]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Stream-aggregate transactions (resumable)        ║
# ╚═══════════════════════════════════════════════════════════╝
TXN_CKPT  = os.path.join(CKPT_DIR, 'ckpt_txn_agg.parquet')
PROG_CKPT = os.path.join(CKPT_DIR, 'ckpt_txn_progress.pkl')

if os.path.exists(FEAT_CKPT):
    print('✅ SKIP — features checkpoint exists.')
elif os.path.exists(TXN_CKPT):
    txn_agg = pd.read_parquet(TXN_CKPT)
    print(f'✅ SKIP — txn_agg loaded {txn_agg.shape}')
else:
    txn_parts = sorted(glob.glob(f'{BASE_PATH}/transactions/**/*.parquet', recursive=True))
    if not txn_parts:
        raise FileNotFoundError('No transaction parquet files found under transactions/.')
    print(f'📥 Streaming {len(txn_parts)} transaction files in batches of {BATCH_SIZE}...')
    if os.path.exists(PROG_CKPT):
        with open(PROG_CKPT,'rb') as f_: prog = pickle.load(f_)
        agg_list, start = prog['agg_list'], prog['next_batch']
        print(f'   Resuming from batch {start}')
    else:
        agg_list, start = [], 0
    COLS = ['account_id','amount','txn_type','channel','transaction_timestamp','counterparty_id']

    for b0 in range(start, len(txn_parts), BATCH_SIZE):
        frames = []
        for fp in txn_parts[b0:b0+BATCH_SIZE]:
            try: frames.append(pd.read_parquet(fp, columns=COLS))
            except Exception as e: print(f'   ⚠️ skip {fp}: {e}')
        if not frames: continue
        bdf = pd.concat(frames, ignore_index=True); del frames; gc.collect()

        bdf['transaction_timestamp'] = pd.to_datetime(bdf['transaction_timestamp'], errors='coerce')
        bdf['hour']   = bdf['transaction_timestamp'].dt.hour.fillna(-1).astype('int8')
        bdf['is_C']   = (bdf['txn_type']=='C').astype('int8')
        bdf['is_D']   = (bdf['txn_type']=='D').astype('int8')
        bdf['is_upi'] = bdf['channel'].isin(['UPC','UPD']).astype('int8')
        bdf['is_round'] = ((bdf['amount']%1000==0)&(bdf['amount']>0)).astype('int8')
        bdf['is_night'] = (((bdf['hour']>=23)|((bdf['hour']>=0)&(bdf['hour']<=4)))&(bdf['hour']>=0)).astype('int8')
        bdf['amt_pos']  = bdf['amount'].clip(lower=0).astype('float32')
        bdf['credit']   = (bdf['amt_pos']*bdf['is_C']).astype('float32')
        bdf['debit']    = (bdf['amt_pos']*bdf['is_D']).astype('float32')

        agg = bdf.groupby('account_id', sort=False).agg(
            txn_count=('amount','count'), total_credit=('credit','sum'), total_debit=('debit','sum'),
            avg_amount=('amount','mean'), max_amount=('amount','max'), std_amount=('amount','std'),
            round_amount_cnt=('is_round','sum'), upi_txn_cnt=('is_upi','sum'),
            night_txn_cnt=('is_night','sum'), unique_channels=('channel','nunique'),
            unique_counterparty=('counterparty_id','nunique')).reset_index()
        for c in agg.select_dtypes('float64').columns: agg[c] = agg[c].astype('float32')
        agg_list.append(agg); del bdf, agg; gc.collect()

        done = min(b0+BATCH_SIZE, len(txn_parts))
        print(f'   Batch {done:>4}/{len(txn_parts)} ({done/len(txn_parts)*100:.0f}%)', end='\r')
        if (b0//BATCH_SIZE) % 5 == 0:
            with open(PROG_CKPT,'wb') as f_:
                pickle.dump({'agg_list':agg_list,'next_batch':done}, f_)
    print()

    print('🔗 Merging batch aggregates...')
    txn_agg = pd.concat(agg_list, ignore_index=True); del agg_list; gc.collect()
    # A credit/debit SUM is additive across batches; counts additive; max via max;
    # avg_amount / std_amount recombined approximately via mean-of-batch (documented limitation).
    txn_agg = txn_agg.groupby('account_id', as_index=False).agg(
        txn_count=('txn_count','sum'), total_credit=('total_credit','sum'),
        total_debit=('total_debit','sum'), avg_amount=('avg_amount','mean'),
        max_amount=('max_amount','max'), std_amount=('std_amount','mean'),
        round_amount_cnt=('round_amount_cnt','sum'), upi_txn_cnt=('upi_txn_cnt','sum'),
        night_txn_cnt=('night_txn_cnt','sum'), unique_channels=('unique_channels','max'),
        unique_counterparty=('unique_counterparty','max'))
    txn_agg.to_parquet(TXN_CKPT, index=False)
    if os.path.exists(PROG_CKPT): os.remove(PROG_CKPT)
    print(f'✅ Transaction aggregation complete: {txn_agg.shape} → saved checkpoint')


✅ SKIP — features checkpoint exists.


In [8]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 4 — Feature engineering (checkpoint-aware)           ║
# ╚═══════════════════════════════════════════════════════════╝
if os.path.exists(FEAT_CKPT):
    master = pd.read_parquet(FEAT_CKPT)
    print(f'✅ SKIP — loaded features {master.shape} | mules {master["is_mule"].sum():,} '
          f'({master["is_mule"].mean()*100:.1f}%)')
else:
    assert 'accounts' in dir(), 'Re-run Cell 2 first.'
    txn_agg = pd.read_parquet(TXN_CKPT)
    print('🔧 Building master feature matrix...')
    df = (accounts.merge(linkage, on='account_id', how='left')
                  .merge(customers, on='customer_id', how='left'))
    demo_safe = demographics.drop(columns=['name','address','phone_number'], errors='ignore')
    df = (df.merge(demo_safe, on='customer_id', how='left')
            .merge(branch, on='branch_code', how='left')
            .merge(products, on='customer_id', how='left')
            .merge(acc_add, on='account_id', how='left')
            .merge(txn_agg, on='account_id', how='left')
            .merge(labels[['account_id','is_mule']], on='account_id', how='inner'))
    del txn_agg; gc.collect()

    REF = pd.Timestamp('2025-07-01')
    for col, new in [('account_opening_date','account_age_days'),
                     ('last_kyc_date','kyc_staleness_days'),
                     ('last_mobile_update_date','mobile_update_recency_days')]:
        if col in df.columns:
            df[new] = (REF - pd.to_datetime(df[col], errors='coerce')).dt.days.astype('float32')
        else:
            df[new] = np.float32(np.nan)

    df['passthrough_ratio'] = (np.minimum(df['total_credit'],df['total_debit']) /
                               (np.maximum(df['total_credit'],df['total_debit'])+1)).fillna(0).astype('float32')
    for cnt, ratio in [('round_amount_cnt','round_amount_ratio'),
                       ('night_txn_cnt','night_txn_ratio'),('upi_txn_cnt','upi_ratio')]:
        df[ratio] = (df[cnt]/(df['txn_count']+1)).astype('float32') if cnt in df.columns else np.float32(0)
    df['balance_range'] = (df['max_amount'] - df['avg_amount'].fillna(0)).astype('float32')

    for col in ['pan_available','aadhaar_available','mobile_banking_flag','kyc_compliant','rural_branch']:
        df[col] = (df[col]=='Y').astype('int8') if col in df.columns else np.int8(0)
    df['is_pmjdy'] = (df['scheme_code']=='PMJDY').astype('int8') if 'scheme_code' in df.columns else np.int8(0)

    for c in df.select_dtypes('float64').columns: df[c] = df[c].astype('float32')
    for c in df.select_dtypes('int64').columns:   df[c] = pd.to_numeric(df[c], downcast='integer')

    KEEP = ['account_id','is_mule','txn_count','total_credit','total_debit','avg_amount',
            'std_amount','max_amount','passthrough_ratio','round_amount_ratio','night_txn_ratio',
            'upi_ratio','unique_channels','unique_counterparty','account_age_days',
            'kyc_staleness_days','mobile_update_recency_days','kyc_compliant','is_pmjdy',
            'rural_branch','avg_balance','monthly_avg_balance','daily_avg_balance',
            'pan_available','aadhaar_available','mobile_banking_flag','loan_count','sa_count',
            'sa_sum','branch_employee_count','balance_range']
    KEEP = [c for c in KEEP if c in df.columns]
    master = df[KEEP].copy(); del df; gc.collect()
    master.to_parquet(FEAT_CKPT, index=False)
    print(f'✅ Features saved {master.shape} | mules {master["is_mule"].sum():,} '
          f'({master["is_mule"].mean()*100:.1f}%)')

for v in ['accounts','customers','demographics','branch','linkage','products','acc_add','labels']:
    globals().pop(v, None)
gc.collect(); print('🧹 Static frames freed')


✅ SKIP — loaded features (96091, 31) | mules 2,683 (2.8%)
🧹 Static frames freed


In [9]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 5 — Test / validation / 3-bank split                 ║
# ║  TEST is touched only for final scoring.                   ║
# ║  VALIDATION (held out from training) is used to pick the   ║
# ║  decision threshold — never the test set.                  ║
# ╚═══════════════════════════════════════════════════════════╝
master = pd.read_parquet(FEAT_CKPT)
FEATURE_COLS = [c for c in master.columns if c not in ('account_id','is_mule')]
X = master[FEATURE_COLS].fillna(0).astype('float32').to_numpy()
y = master['is_mule'].astype(int).to_numpy()
del master; gc.collect()

# 1) hold out final test (20%)
X_tr, X_test, y_tr, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
# 2) hold out validation for threshold selection (15% of remainder)
X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=42)
# 3) partition the rest into 3 banks of unequal size (40 / 35 / 25)
idx = np.random.RandomState(42).permutation(len(X_tr))
s1, s2 = int(0.40*len(idx)), int(0.75*len(idx))
BANKS = [('Bank A', X_tr[idx[:s1]],  y_tr[idx[:s1]],  COLORS['bankA']),
         ('Bank B', X_tr[idx[s1:s2]],y_tr[idx[s1:s2]],COLORS['bankB']),
         ('Bank C', X_tr[idx[s2:]],  y_tr[idx[s2:]],  COLORS['bankC'])]

print('Partition (mule rate):')
for nm,Xb,yb,_ in BANKS: print(f'  {nm}: {len(Xb):>7,} | {yb.mean()*100:.2f}%')
print(f'  {"Validation":<6}: {len(X_val):>7,} | {y_val.mean()*100:.2f}%')
print(f'  {"Test":<6}: {len(X_test):>7,} | {y_test.mean()*100:.2f}%')
print(f'  Features : {len(FEATURE_COLS)}')


Partition (mule rate):
  Bank A:  26,136 | 2.97%
  Bank B:  22,869 | 2.70%
  Bank C:  16,336 | 2.64%
  Validation:  11,531 | 2.79%
  Test  :  19,219 | 2.79%
  Features : 29


### Cell 6 — Federated primitives
Everything here is deliberately transparent (plain NumPy) so the federation is auditable:

- **`federated_scaler`** — each bank shares only `count, sum, sum-of-squares` per feature (not rows). These aggregate to a global mean/std. Standardisation without moving raw data.
- **`local_update`** — a bank runs a few epochs of class-weighted logistic-regression gradient descent **starting from the current global weights**.
- **`fedavg`** — the server takes each bank's weight *update*, (optionally clips + adds DP noise), averages them by data volume, and **carries the new global weights into the next round**. This is the real FedAvg loop the earlier notebook lacked.
- **`pick_threshold`** — choose the score cut-off on the **validation** set at a target recall, so precision/F1 aren't gamed on the test set.

In [10]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Federated primitives (auditable NumPy)           ║
# ╚═══════════════════════════════════════════════════════════╝
TARGET_RECALL = 0.80     # operating point: catch 80% of mules, then report precision there
ROUNDS        = 15
LOCAL_EPOCHS  = 5
LR            = 0.4

def _sigmoid(z): return 1.0/(1.0+np.exp(-np.clip(z,-30,30)))
def _bias(Xs):   return np.hstack([Xs, np.ones((len(Xs),1))])

def federated_scaler(banks):
    """Global mean/std from shared sufficient statistics only (no raw rows)."""
    d = banks[0][1].shape[1]; N=0; S=np.zeros(d); Q=np.zeros(d)
    for _,Xb,_,_ in banks:
        Xb = Xb.astype('float64'); N += len(Xb); S += Xb.sum(0); Q += (Xb**2).sum(0)
    mu = S/N; var = np.maximum(Q/N - mu**2, 1e-8)
    return mu, np.sqrt(var)

def local_update(w, Xs, yb, epochs, lr, pos_weight, l2=1e-4):
    """Local class-weighted logistic GD from global w; returns updated local w."""
    w = w.copy(); n = len(yb); sw = np.where(yb==1, pos_weight, 1.0); Xb = _bias(Xs)
    for _ in range(epochs):
        p = _sigmoid(Xb @ w)
        g = Xb.T @ (sw*(p-yb))/n + l2*w
        w -= lr*g
    return w

def fedavg(banks, rounds=ROUNDS, local_epochs=LOCAL_EPOCHS, lr=LR,
           dp_sigma=0.0, clip=1.0, seed=0, X_eval=None, y_eval=None, mu=None, sd=None):
    """Real FedAvg. dp_sigma>0 → clip each client update to `clip`, add Gaussian noise."""
    if mu is None: mu, sd = federated_scaler(banks)
    Xs = [ (Xb-mu)/sd for _,Xb,_,_ in banks ]; ys = [ yb for _,_,yb,_ in banks ]
    ns = np.array([len(v) for v in ys], float); wts = ns/ns.sum()
    pos_weight = sum((v==0).sum() for v in ys)/max(sum((v==1).sum() for v in ys),1)
    d = Xs[0].shape[1]+1; r = np.random.RandomState(seed); w = r.randn(d)*0.01
    hist = []
    Xe = None if X_eval is None else (X_eval-mu)/sd
    for rnd in range(1, rounds+1):
        deltas = []
        for Xb, yb in zip(Xs, ys):
            dl = local_update(w, Xb, yb, local_epochs, lr, pos_weight) - w
            if dp_sigma > 0:
                dl = dl / max(1.0, np.linalg.norm(dl)/clip)       # per-client norm clip
            deltas.append(dl)
        agg = sum(a*dl for a,dl in zip(wts, deltas))
        if dp_sigma > 0:
            agg = agg + r.normal(0, dp_sigma*clip/len(banks), size=agg.shape)  # calibrated noise
        w = w + agg
        if Xe is not None:
            p = _sigmoid(_bias(Xe) @ w)
            hist.append({'round':rnd,'auc':roc_auc_score(y_eval,p),
                         'auprc':average_precision_score(y_eval,p)})
    return w, (mu, sd), hist

def proba(w, Xs, mu, sd): return _sigmoid(_bias((Xs-mu)/sd) @ w)

def pick_threshold(p_val, y_val, target_recall=TARGET_RECALL):
    o = np.argsort(-p_val); ys = y_val[o]; ps = p_val[o]
    rec = np.cumsum(ys)/max(ys.sum(),1)
    i = min(int(np.searchsorted(rec, target_recall)), len(ps)-1)
    return float(ps[i])

def scores(p, y, thr):
    pred = (p>=thr).astype(int)
    return dict(auc=roc_auc_score(y,p), auprc=average_precision_score(y,p),
                f1=f1_score(y,pred,zero_division=0),
                precision=precision_score(y,pred,zero_division=0),
                recall=recall_score(y,pred,zero_division=0),
                flag_rate=pred.mean())
print(f'✅ Primitives ready | operating point = recall≈{TARGET_RECALL:.0%}, '
      f'{ROUNDS} rounds × {LOCAL_EPOCHS} local epochs')


✅ Primitives ready | operating point = recall≈80%, 15 rounds × 5 local epochs


In [11]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Local-only baselines (what each bank has today)  ║
# ║  Same logistic model class as the federated system → fair. ║
# ╚═══════════════════════════════════════════════════════════╝
RESULTS = {}            # name -> metric dict (on TEST, at validation-chosen threshold)
local_probas = {}

for nm, Xb, yb, _ in BANKS:
    mu_l, sd_l = Xb.astype('float64').mean(0), Xb.astype('float64').std(0)+1e-8
    pw = (yb==0).sum()/max((yb==1).sum(),1)
    w  = np.random.RandomState(0).randn(Xb.shape[1]+1)*0.01
    for _ in range(ROUNDS*LOCAL_EPOCHS):                 # same total gradient steps as FedAvg
        w = local_update(w, (Xb-mu_l)/sd_l, yb, 1, LR, pw)
    thr = pick_threshold(proba(w, X_val, mu_l, sd_l), y_val)
    p_test = proba(w, X_test, mu_l, sd_l)
    RESULTS[f'{nm} (local only)'] = scores(p_test, y_test, thr)
    local_probas[nm] = p_test

print(f'{"Model":<22}{"AUC":>7}{"AUPRC":>8}{"Prec":>7}{"Recall":>8}{"F1":>7}{"Flag%":>7}')
for k,v in RESULTS.items():
    print(f'{k:<22}{v["auc"]:>7.3f}{v["auprc"]:>8.3f}{v["precision"]:>7.3f}'
          f'{v["recall"]:>8.3f}{v["f1"]:>7.3f}{v["flag_rate"]*100:>6.1f}%')


Model                     AUC   AUPRC   Prec  Recall     F1  Flag%
Bank A (local only)     0.885   0.282  0.159   0.747  0.262  13.1%
Bank B (local only)     0.888   0.272  0.143   0.780  0.242  15.2%
Bank C (local only)     0.885   0.288  0.121   0.799  0.210  18.4%


In [12]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 8 — Train FedShield (real FedAvg) + seed stability   ║
# ╚═══════════════════════════════════════════════════════════╝
N_SEEDS = 5
mu_g, sd_g = federated_scaler(BANKS)        # shared global scaler (stats only)

# Convergence history from seed 0 (evaluated each round on VALIDATION)
w_fed, _, hist = fedavg(BANKS, seed=0, X_eval=X_val, y_eval=y_val, mu=mu_g, sd=sd_g)
fed_hist = pd.DataFrame(hist)
print('Real FedAvg convergence on validation (monotone, not seed-jitter):')
for h in hist[::max(1,len(hist)//8)]:
    print(f'  round {h["round"]:>2}: AUC={h["auc"]:.4f}  AUPRC={h["auprc"]:.4f}')

# Stability: run the whole FL process over several seeds
seed_rows = []
for s in range(N_SEEDS):
    ws, _, _ = fedavg(BANKS, seed=s, mu=mu_g, sd=sd_g)
    thr_s = pick_threshold(proba(ws, X_val, mu_g, sd_g), y_val)
    seed_rows.append(scores(proba(ws, X_test, mu_g, sd_g), y_test, thr_s))
seed_df = pd.DataFrame(seed_rows)

# Headline federated result = mean over seeds (with std); threshold from seed-0 model
thr_fed   = pick_threshold(proba(w_fed, X_val, mu_g, sd_g), y_val)
p_fed_test = proba(w_fed, X_test, mu_g, sd_g)
fed_mean  = seed_df.mean().to_dict(); fed_std = seed_df.std().to_dict()
RESULTS['FedShield (FedAvg)'] = scores(p_fed_test, y_test, thr_fed)

print(f'\nFedShield over {N_SEEDS} seeds (mean ± std):')
for m in ['auc','auprc','precision','recall','f1']:
    print(f'  {m:<9}: {fed_mean[m]:.3f} ± {fed_std[m]:.3f}')


Real FedAvg convergence on validation (monotone, not seed-jitter):
  round  1: AUC=0.8831  AUPRC=0.3413
  round  2: AUC=0.8847  AUPRC=0.3505
  round  3: AUC=0.8855  AUPRC=0.3545
  round  4: AUC=0.8861  AUPRC=0.3572
  round  5: AUC=0.8864  AUPRC=0.3601
  round  6: AUC=0.8867  AUPRC=0.3626
  round  7: AUC=0.8870  AUPRC=0.3643
  round  8: AUC=0.8872  AUPRC=0.3658
  round  9: AUC=0.8874  AUPRC=0.3663
  round 10: AUC=0.8875  AUPRC=0.3672
  round 11: AUC=0.8877  AUPRC=0.3680
  round 12: AUC=0.8878  AUPRC=0.3693
  round 13: AUC=0.8879  AUPRC=0.3704
  round 14: AUC=0.8881  AUPRC=0.3718
  round 15: AUC=0.8882  AUPRC=0.3726

FedShield over 5 seeds (mean ± std):
  auc      : 0.888 ± 0.000
  auprc    : 0.282 ± 0.000
  precision: 0.146 ± 0.000
  recall   : 0.778 ± 0.000
  f1       : 0.246 ± 0.000


In [13]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 9 — Centralised references (privacy-free upper bound)║
# ║  What you COULD get if raw data were pooled (RBI forbids). ║
# ╚═══════════════════════════════════════════════════════════╝
import lightgbm as lgb

X_pool = np.vstack([Xb for _,Xb,_,_ in BANKS]); y_pool = np.concatenate([yb for _,_,yb,_ in BANKS])

# (a) Centralised logistic — same class as federated, so gap = pure cost of privacy
pw = (y_pool==0).sum()/max((y_pool==1).sum(),1)
w_cen = np.random.RandomState(0).randn(X_pool.shape[1]+1)*0.01
for _ in range(ROUNDS*LOCAL_EPOCHS):
    w_cen = local_update(w_cen, (X_pool-mu_g)/sd_g, y_pool, 1, LR, pw)
thr_cen = pick_threshold(proba(w_cen, X_val, mu_g, sd_g), y_val)
RESULTS['Centralised LogReg'] = scores(proba(w_cen, X_test, mu_g, sd_g), y_test, thr_cen)

# (b) Centralised LightGBM — strong non-linear ceiling + used for SHAP later
lgbm = lgb.LGBMClassifier(objective='binary', n_estimators=300, learning_rate=0.05,
                          num_leaves=31, max_depth=6, min_child_samples=100,
                          subsample=0.8, colsample_bytree=0.8,
                          scale_pos_weight=pw, n_jobs=2, verbose=-1, random_state=42)
lgbm.fit(X_pool, y_pool)
p_lgb_val  = lgbm.predict_proba(X_val)[:,1]
p_lgb_test = lgbm.predict_proba(X_test)[:,1]
thr_lgb    = pick_threshold(p_lgb_val, y_val)
RESULTS['Centralised LightGBM'] = scores(p_lgb_test, y_test, thr_lgb)

for k in ['Centralised LogReg','Centralised LightGBM']:
    v = RESULTS[k]
    print(f'{k:<22} AUC={v["auc"]:.3f} AUPRC={v["auprc"]:.3f} '
          f'P={v["precision"]:.3f} R={v["recall"]:.3f} F1={v["f1"]:.3f}')


Centralised LogReg     AUC=0.888 AUPRC=0.281 P=0.145 R=0.778 F1=0.244
Centralised LightGBM   AUC=0.932 AUPRC=0.643 P=0.603 R=0.777 F1=0.679


In [14]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 10 — Differential privacy: measured utility cost     ║
# ╚═══════════════════════════════════════════════════════════╝
dp_rows = []
for sigma in [0.0, 0.25, 0.5, 1.0, 2.0]:
    wdp, _, _ = fedavg(BANKS, dp_sigma=sigma, clip=1.0, seed=0, mu=mu_g, sd=sd_g)
    thr = pick_threshold(proba(wdp, X_val, mu_g, sd_g), y_val)
    m = scores(proba(wdp, X_test, mu_g, sd_g), y_test, thr); m['dp_sigma'] = sigma
    dp_rows.append(m)
dp_df = pd.DataFrame(dp_rows)[['dp_sigma','auc','auprc','precision','recall','f1']]
print('Noise σ  (0 = no privacy noise):')
print(dp_df.round(3).to_string(index=False))
print('\n→ Higher σ = stronger privacy, lower utility. Pick σ on the knee of this curve.')


Noise σ  (0 = no privacy noise):
 dp_sigma   auc  auprc  precision  recall    f1
     0.00 0.888  0.282      0.146   0.778 0.246
     0.25 0.864  0.247      0.101   0.797 0.179
     0.50 0.824  0.189      0.075   0.803 0.138
     1.00 0.752  0.115      0.048   0.791 0.091
     2.00 0.649  0.074      0.033   0.790 0.063

→ Higher σ = stronger privacy, lower utility. Pick σ on the knee of this curve.


In [15]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 11 — Results summary + CSV export                    ║
# ╚═══════════════════════════════════════════════════════════╝
order = [f'{b[0]} (local only)' for b in BANKS] + \
        ['FedShield (FedAvg)','Centralised LogReg','Centralised LightGBM']
print('='*78)
print('                 FedShield — RESULTS (TEST set, threshold set on validation)')
print('='*78)
print(f'{"Model":<24}{"AUC":>7}{"AUPRC":>8}{"Prec":>7}{"Recall":>8}{"F1":>7}{"Flag%":>7}')
print('-'*78)
for k in order:
    v = RESULTS[k]
    print(f'{k:<24}{v["auc"]:>7.3f}{v["auprc"]:>8.3f}{v["precision"]:>7.3f}'
          f'{v["recall"]:>8.3f}{v["f1"]:>7.3f}{v["flag_rate"]*100:>6.1f}%')
print('='*78)

best_local_auprc = max(RESULTS[f'{b[0]} (local only)']['auprc'] for b in BANKS)
fed = RESULTS['FedShield (FedAvg)']; cen = RESULTS['Centralised LightGBM']
print(f'\nAUPRC — primary metric at {y_test.mean()*100:.1f}% prevalence:')
print(f'  best local-only : {best_local_auprc:.3f}')
print(f'  FedShield       : {fed["auprc"]:.3f}  ({fed["auprc"]-best_local_auprc:+.3f} vs best local)')
print(f'  centralised LGBM: {cen["auprc"]:.3f}  (privacy-free ceiling)')
print(f'\nFedShield recovers '
      f'{(fed["auprc"]-best_local_auprc)/max(cen["auprc"]-best_local_auprc,1e-9)*100:.0f}% '
      f'of the local→centralised gap, with zero raw-data sharing.')

summary = pd.DataFrame([{'model':k, **{m:round(RESULTS[k][m],4)
                 for m in ['auc','auprc','precision','recall','f1','flag_rate']}} for k in order])
summary.to_csv(f'{CKPT_DIR}/fedshield_results.csv', index=False)
fed_hist.to_csv(f'{CKPT_DIR}/fedshield_convergence.csv', index=False)
seed_df.to_csv(f'{CKPT_DIR}/fedshield_seed_stability.csv', index=False)
dp_df.to_csv(f'{CKPT_DIR}/fedshield_dp_sweep.csv', index=False)
print('\n✅ CSVs saved to', CKPT_DIR)
summary


                 FedShield — RESULTS (TEST set, threshold set on validation)
Model                       AUC   AUPRC   Prec  Recall     F1  Flag%
------------------------------------------------------------------------------
Bank A (local only)       0.885   0.282  0.159   0.747  0.262  13.1%
Bank B (local only)       0.888   0.272  0.143   0.780  0.242  15.2%
Bank C (local only)       0.885   0.288  0.121   0.799  0.210  18.4%
FedShield (FedAvg)        0.888   0.282  0.146   0.778  0.246  14.9%
Centralised LogReg        0.888   0.281  0.145   0.778  0.244  15.0%
Centralised LightGBM      0.932   0.643  0.603   0.777  0.679   3.6%

AUPRC — primary metric at 2.8% prevalence:
  best local-only : 0.288
  FedShield       : 0.282  (-0.006 vs best local)
  centralised LGBM: 0.643  (privacy-free ceiling)

FedShield recovers -2% of the local→centralised gap, with zero raw-data sharing.

✅ CSVs saved to /kaggle/working


,model,auc,auprc,precision,recall,f1,flag_rate
0,Bank A (local only),0.8847,0.2816,0.1591,0.7467,0.2623,0.1311
1,Bank B (local only),0.8880,0.2721,0.1432,0.7803,0.2421,0.1522
2,Bank C (local only),0.8852,0.2878,0.1211,0.7989,0.2103,0.1843
3,FedShield (FedAvg),0.8878,0.2817,0.1460,0.7784,0.2459,0.1490
4,Centralised LogReg,0.8879,0.2814,0.1449,0.7784,0.2443,0.1501
5,Centralised LightGBM,0.9321,0.6425,0.6026,0.7765,0.6786,0.0360


In [16]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 12 — Visualisations                                  ║
# ╚═══════════════════════════════════════════════════════════╝
def _ax(ax):
    ax.set_facecolor('#1a1a2e')
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
    for sp in ['bottom','left']: ax.spines[sp].set_color('#333')
    ax.tick_params(colors='#ccc')

def save(name):
    plt.savefig(f'{CKPT_DIR}/{name}', dpi=110, bbox_inches='tight', facecolor='#0d1117')
    plt.show(); print('✅', name)

# (1) convergence + seed stability
fig, ax = plt.subplots(1, 2, figsize=(13,4.5)); fig.patch.set_facecolor('#0d1117')
_ax(ax[0])
ax[0].plot(fed_hist['round'], fed_hist['auc'],   color=COLORS['green'], marker='o', ms=4, label='AUC')
ax[0].plot(fed_hist['round'], fed_hist['auprc'], color=COLORS['gold'],  marker='s', ms=4, label='AUPRC')
ax[0].set_title('Real FedAvg convergence (validation)', color='white', fontweight='bold')
ax[0].set_xlabel('round', color='#ccc'); ax[0].set_ylim(0,1)
ax[0].legend(labelcolor='#ccc', facecolor='#1a1a2e', edgecolor='#333')
_ax(ax[1])
mets=['auc','auprc','precision','recall','f1']
ax[1].bar(mets, [seed_df[m].mean() for m in mets], yerr=[seed_df[m].std() for m in mets],
          color=COLORS['fed']+'99', edgecolor=COLORS['fed'], capsize=4)
ax[1].set_title(f'FedShield stability (mean±std, {N_SEEDS} seeds)', color='white', fontweight='bold')
ax[1].set_ylim(0,1)
plt.tight_layout(); save('plot_convergence_stability.png')

# (2) local vs federated vs centralised  — grouped bars by metric
fig, ax = plt.subplots(figsize=(11,4.5)); fig.patch.set_facecolor('#0d1117'); _ax(ax)
order = [f'{b[0]} (local only)' for b in BANKS]+['FedShield (FedAvg)','Centralised LogReg','Centralised LightGBM']
plot_metrics = [('auprc',COLORS['gold']),('precision',COLORS['green']),('recall',COLORS['red'])]
x = np.arange(len(order)); w = 0.26
for i,(m,c) in enumerate(plot_metrics):
    ax.bar(x+(i-1)*w, [RESULTS[k][m] for k in order], w, label=m.upper(), color=c+'cc', edgecolor=c)
ax.set_xticks(x); ax.set_xticklabels([k.replace(' (','\n(') for k in order], color='#ccc', fontsize=8)
ax.set_ylim(0,1); ax.set_title('Local-only vs FedShield vs Centralised', color='white', fontweight='bold')
ax.legend(labelcolor='#ccc', facecolor='#1a1a2e', edgecolor='#333')
plt.tight_layout(); save('plot_local_vs_fed.png')

# (3) PR curves
fig, ax = plt.subplots(figsize=(7,5.5)); fig.patch.set_facecolor('#0d1117'); _ax(ax)
for nm,_,_,c in BANKS:
    pr,rc,_ = precision_recall_curve(y_test, local_probas[nm]); ax.plot(rc,pr,color=c,alpha=.7,lw=1.5,label=f'{nm} local')
pr,rc,_ = precision_recall_curve(y_test, p_fed_test); ax.plot(rc,pr,color=COLORS['fed'],lw=2.5,label='FedShield')
pr,rc,_ = precision_recall_curve(y_test, p_lgb_test); ax.plot(rc,pr,color=COLORS['blue'],lw=2,ls='--',label='Centralised LGBM')
ax.axhline(y_test.mean(), color='#888', ls=':', lw=1, label=f'prevalence {y_test.mean()*100:.1f}%')
ax.set_xlabel('Recall', color='#ccc'); ax.set_ylabel('Precision', color='#ccc')
ax.set_title('Precision–Recall (test)', color='white', fontweight='bold')
ax.legend(fontsize=8, labelcolor='#ccc', facecolor='#1a1a2e', edgecolor='#333')
plt.tight_layout(); save('plot_pr_curves.png')

# (4) DP tradeoff
fig, ax = plt.subplots(figsize=(7,4.5)); fig.patch.set_facecolor('#0d1117'); _ax(ax)
ax.plot(dp_df['dp_sigma'], dp_df['auc'],   marker='o', color=COLORS['green'], label='AUC')
ax.plot(dp_df['dp_sigma'], dp_df['auprc'], marker='s', color=COLORS['gold'],  label='AUPRC')
ax.set_xlabel('DP noise σ', color='#ccc'); ax.set_ylabel('score', color='#ccc'); ax.set_ylim(0,1)
ax.set_title('Privacy / utility tradeoff', color='white', fontweight='bold')
ax.legend(labelcolor='#ccc', facecolor='#1a1a2e', edgecolor='#333')
plt.tight_layout(); save('plot_dp_tradeoff.png')


✅ plot_convergence_stability.png
✅ plot_local_vs_fed.png
✅ plot_pr_curves.png
✅ plot_dp_tradeoff.png


In [17]:
# ╔═══════════════════════════════════════════════════════════╗
# ║  CELL 13 — Explainability                                  ║
# ║  (a) federated model's own coefficients (exact, linear)    ║
# ║  (b) SHAP on the centralised LightGBM reference            ║
# ╚═══════════════════════════════════════════════════════════╝
import shap

# (a) Federated logistic coefficients — directly interpretable, no approximation
coef = w_fed[:-1] * sd_g            # undo standardisation scale → per-unit weight
imp = pd.DataFrame({'feature':FEATURE_COLS,'weight':w_fed[:-1]}) \
        .assign(abs=lambda d:d['weight'].abs()).sort_values('abs',ascending=False)
print('Top drivers in the FEDERATED model (logistic weights, standardised):')
for _,r in imp.head(10).iterrows():
    arrow = '↑ raises risk' if r['weight']>0 else '↓ lowers risk'
    print(f'  {r["feature"]:<26}{r["weight"]:+.3f}  {arrow}')

fig, ax = plt.subplots(figsize=(8,5)); fig.patch.set_facecolor('#0d1117'); _ax(ax)
top = imp.head(12).iloc[::-1]
ax.barh(top['feature'], top['weight'],
        color=[COLORS['red'] if w>0 else COLORS['blue'] for w in top['weight']])
ax.set_title('Federated model — feature weights', color='white', fontweight='bold')
ax.set_xlabel('logistic weight (standardised)', color='#ccc')
plt.tight_layout(); save('plot_fed_coefficients.png')

# (b) SHAP on centralised LightGBM (tree explainer, 300-row sample for memory)
N = min(300, len(X_test))
sel = np.random.RandomState(42).choice(len(X_test), N, replace=False)
Xs = pd.DataFrame(X_test[sel], columns=FEATURE_COLS)
sv = shap.TreeExplainer(lgbm).shap_values(Xs)
sv = sv[1] if isinstance(sv, list) else sv
plt.figure(figsize=(9,6)); plt.gcf().set_facecolor('#0d1117'); plt.gca().set_facecolor('#1a1a2e')
shap.summary_plot(sv, Xs, plot_type='bar', show=False, color=COLORS['gold'])
plt.title('SHAP — centralised LightGBM reference', color='white')
plt.tick_params(colors='#ccc'); save('plot_shap_lightgbm.png')

# Sample alert explanation (federated score + LightGBM SHAP reasons)
mu_idx = np.where(y_test[sel]==1)[0]
if len(mu_idx):
    i = mu_idx[0]; risk = p_fed_test[sel[i]]
    print(f'\n🚨 ALERT — true MULE | FedShield risk score {risk:.1%} '
          f'→ {"BLOCK" if risk>0.7 else "REVIEW" if risk>thr_fed else "pass"}')
    top5 = sorted(zip(FEATURE_COLS, sv[i]), key=lambda t:abs(t[1]), reverse=True)[:5]
    for f,s in top5:
        print(f'   • {f:<26} SHAP {s:+.3f}  {"↑ increases" if s>0 else "↓ decreases"} risk')


Top drivers in the FEDERATED model (logistic weights, standardised):
  unique_counterparty       +1.164  ↑ raises risk
  kyc_staleness_days        -0.653  ↓ lowers risk
  unique_channels           -0.506  ↓ lowers risk
  txn_count                 -0.489  ↓ lowers risk
  total_debit               +0.363  ↑ raises risk
  total_credit              +0.281  ↑ raises risk
  night_txn_ratio           +0.258  ↑ raises risk
  upi_ratio                 -0.218  ↓ lowers risk
  account_age_days          -0.190  ↓ lowers risk
  branch_employee_count     -0.171  ↓ lowers risk
✅ plot_fed_coefficients.png
✅ plot_shap_lightgbm.png

🚨 ALERT — true MULE | FedShield risk score 98.3% → BLOCK
   • unique_counterparty        SHAP +2.239  ↑ increases risk
   • avg_amount                 SHAP -1.626  ↓ decreases risk
   • std_amount                 SHAP +0.704  ↑ increases risk
   • kyc_staleness_days         SHAP -0.192  ↓ decreases risk
   • round_amount_ratio         SHAP +0.174  ↑ increases risk


## Scope, limitations & what a production pilot needs

**Honestly implemented here**
- Real **FedAvg** (global weights, carried across rounds) — not probability averaging.
- **Federated standardisation** from shared statistics only; no raw rows leave a "bank".
- **Differential privacy** via per-client update clipping + calibrated Gaussian noise, with a measured utility curve.
- Fair **local-only** baselines (same model class) and **centralised** upper bounds (LogReg + LightGBM).
- Threshold chosen on **validation** at a target recall; results reported as **mean ± std over seeds**; **AUPRC** as the headline metric for the 2.8%-prevalence setting.

**Simulated / not implemented (do not claim these as results)**
- The "3 banks" are one dataset partitioned, not three institutions. Cross-bank benefit depends on how non-IID the real partitions are.
- No live **EFRM / AML / I4C / DPIP** feeds; the 7-layer evidence-fusion, Byzantine validation, and living-pattern-library layers are architecture, not code in this notebook.
- The logistic federated model trades some accuracy for the clean, averageable weights FedAvg needs; the LightGBM row shows the non-linear ceiling if you later move to federated gradient-boosting (e.g. via Flower + a tree-FL scheme).
- `std_amount` / `avg_amount` are recombined approximately across transaction batches (noted in Cell 3); exact streaming variance (Welford) is a drop-in improvement.

**To turn this into a pilot**: two real banks + a neutral aggregator (RBI/NPCI), per-bank local preprocessing, a fixed DP budget (ε) agreed up front, and a recall/precision operating point signed off by fraud-ops before go-live.